# Pasado `gpu-hybrid`: Colab merenje (`--n-splits = 1..64` + VRAM + Cubic Fixes)

Pre pokretanja izaberi **Runtime > Change runtime type > GPU** (poželjno A100).

Ova sveska izvršava sve na pravom CUDA GPU-u:
1. **Test obe Sašine ideje za kompleksni kubni solver na CUDA-i** (`verify_complex_cubic_fixes.py`):
   - Ideja 1: fiksni prag (`1e-6`) + provera uvrštavanjem u kubnu jednačinu `2*r^3 - 3*r^2 + r - y = 0`
   - Ideja 2: **Miné (2004)** automatska granica zaokruživanja (`delta_imag = 8.26e-7`, `delta_res`)
   - Poređenje sa starim `torch.isreal` i sa `Viète` realnim solverom na sve 4 mreže.
2. **CUDA `--n-splits` sweep (`1, 2, 4, 8, 16, 32, 64`)** na sve 4 mreže (`3layer`, `4layer`, `5layer`, `big`), uz merenje:
   - `lc_precise` i normalizovanog bounda `LC(n) / LC(1)`,
   - vremena izvršavanja (`time_median_ms`) i vremena po splitu (`ms_per_split`),
   - **maksimalnog zauzeća GPU memorije (`peak_vram_mb`)** po broju splitova.
3. **Generisanje čistog 2x2 dual-axis grafika** (`/content/split_sweep_dual_axis_cuda_to64.png`) i automatsko preuzimanje zip arhive.


In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Uključi GPU: Runtime > Change runtime type > GPU"

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


In [ ]:
import os
import shutil
import subprocess

REPO = "/content/Pasado"
URL = "https://github.com/dimitrijepesic/Pasado.git"

shutil.rmtree(REPO, ignore_errors=True)
clone = subprocess.run([
    "git", "-c", "http.version=HTTP/1.1", "clone",
    "--branch", "gpu-hybrid", "--single-branch", URL, REPO,
], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(clone.stdout)

head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
print("HEAD:", head)
os.chdir(REPO)


In [ ]:
%pip install -q scikit-learn


In [ ]:
%%writefile /content/Pasado/experiments/measure_split_sweep.py
"""Measures the precision vs. runtime trade-off of interval splitting (--n-splits).

For each network, image, epsilon index, and n_splits value (default: 1, 2, 4, 8, 16),
this script divides [0, epsilon] into n_splits equal sub-intervals:
  [i * delta, (i + 1) * delta],  delta = epsilon / n_splits,
evaluates the precise forward pass on each sub-interval, and takes the maximum
local Lipschitz bound across the sub-intervals.

It records:
  - lc_precise:           bound for (network, image, epsilon, n_splits)
  - lc_split1:            baseline bound at n_splits = 1
  - bound_reduction_pct:  (lc_split1 - lc_precise) / lc_split1 * 100
  - tightening_factor:    lc_split1 / lc_precise
  - time_median_ms:       median wall-clock time (ms) across measured repeats
  - time_min_ms, time_max_ms
  - time_multiplier:      time_median_ms / time_split1_median_ms

Run:
  python experiments/measure_split_sweep.py --device cuda --networks 3layer big --out split_sweep.csv
  python experiments/measure_split_sweep.py --device mps --networks 3layer
"""
import argparse
import csv
import os
import statistics
import sys
import time

import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, os.path.join(ROOT, "forward_mode_tensorized_src"))
sys.path.insert(0, os.path.join(ROOT, "Section_5_4"))

CSV_COLUMNS = [
    "network",
    "device",
    "dtype",
    "image_id",
    "epsilon_index",
    "epsilon",
    "n_splits",
    "lc_precise",
    "lc_split1",
    "bound_reduction_pct",
    "tightening_factor",
    "time_median_ms",
    "time_min_ms",
    "time_max_ms",
    "time_split1_median_ms",
    "time_multiplier",
    "per_split_ms",
    "peak_vram_mb",
]


def positive_int(value):
    parsed = int(value)
    if parsed < 1:
        raise argparse.ArgumentTypeError("must be at least 1")
    return parsed


def non_negative_int(value):
    parsed = int(value)
    if parsed < 0:
        raise argparse.ArgumentTypeError("must be at least 0")
    return parsed


def numpy_seed(value):
    parsed = int(value)
    if not 0 <= parsed < 2 ** 32:
        raise argparse.ArgumentTypeError("must be between 0 and 2**32 - 1")
    return parsed


def parse_args():
    ap = argparse.ArgumentParser(
        description="Measure precision vs. runtime trade-off across --n-splits values"
    )
    ap.add_argument(
        "--networks",
        "--network",
        dest="networks",
        nargs="+",
        choices=["3layer", "4layer", "5layer", "big"],
        default=["3layer", "big"],
        help="networks to evaluate; default: 3layer big",
    )
    ap.add_argument(
        "--device",
        default=os.environ.get("PASADO_DEVICE", "cuda"),
        help="torch device (cuda, mps, cpu); default: PASADO_DEVICE or cuda",
    )
    ap.add_argument(
        "--dtype",
        choices=["float32", "float64"],
        default=None,
        help="floating-point dtype; default: float32 on mps, float64 otherwise",
    )
    ap.add_argument(
        "--splits",
        type=positive_int,
        nargs="+",
        default=[1, 2, 4, 8, 16],
        help="list of n_splits values to evaluate; default: 1 2 4 8 16",
    )
    ap.add_argument(
        "--num-images",
        type=positive_int,
        default=3,
        help="number of correctly classified test images; default: 3",
    )
    ap.add_argument(
        "--eps-indices",
        type=int,
        nargs="+",
        default=[0, 8, 15],
        help="indices into the 16-epsilon schedule; default: 0 8 15",
    )
    ap.add_argument("--seed", type=numpy_seed, default=7)
    ap.add_argument(
        "--warmup",
        type=non_negative_int,
        default=2,
        help="number of unmeasured warm-up runs per configuration; default: 2",
    )
    ap.add_argument(
        "--repeats",
        type=positive_int,
        default=3,
        help="number of timed repeats per (image, epsilon, n_splits); default: 3",
    )
    ap.add_argument(
        "--real-cubic",
        dest="real_cubic",
        action="store_true",
        default=True,
        help="use the real Viete cubic root solver (default)",
    )
    ap.add_argument(
        "--no-real-cubic",
        dest="real_cubic",
        action="store_false",
        help="use the legacy complex Cardano cubic root solver",
    )
    ap.add_argument(
        "--out",
        default=None,
        help="optional path to write CSV measurements",
    )
    args = ap.parse_args()

    if 1 not in args.splits:
        args.splits = [1] + list(args.splits)
    if len(set(args.eps_indices)) != len(args.eps_indices):
        ap.error("--eps-indices must not contain duplicates")
    invalid_eps = [i for i in args.eps_indices if not 0 <= i < 16]
    if invalid_eps:
        ap.error(f"--eps-indices entries must be between 0 and 15: {invalid_eps}")

    try:
        device = torch.device(args.device)
    except (RuntimeError, ValueError) as error:
        ap.error(f"invalid --device {args.device!r}: {error}")
    if device.type == "cuda" and not torch.cuda.is_available():
        sys.exit("--device cuda was requested but CUDA is not available")
    if device.type == "mps" and not torch.backends.mps.is_available():
        sys.exit("--device mps was requested but MPS is not available")

    default_dtype = "float32" if device.type == "mps" else "float64"
    dtype_name = args.dtype or default_dtype
    if device.type == "mps" and dtype_name == "float64":
        ap.error("MPS does not support float64; use --dtype float32")

    return args, device, dtype_name, getattr(torch, dtype_name)


def sync_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def stamp(device):
    sync_device(device)
    return time.perf_counter()


def load_network_and_images(network_name, device, dtype, num_images):
    import torchvision
    import torchvision.transforms as transforms
    from model import FCN, FCNBig

    section_dir = os.path.join(ROOT, "Section_5_4")
    if network_name == "3layer":
        net = FCN(3)
    elif network_name == "4layer":
        net = FCN(4)
    elif network_name == "5layer":
        net = FCN(5)
    elif network_name == "big":
        net = FCNBig()
    else:
        raise ValueError(f"unknown network: {network_name}")

    weights_path = os.path.join(section_dir, "trained", f"model_{network_name}.pth")
    net.load_state_dict(torch.load(weights_path, map_location="cpu"))
    net = net.eval().to(dtype=dtype, device=device)

    indices_path = os.path.join(section_dir, "trained", f"indices_{network_name}.pth")
    correct_indices = {
        int(index)
        for index in torch.load(indices_path, map_location="cpu")
    }
    if num_images > len(correct_indices):
        raise RuntimeError(
            f"--num-images {num_images} exceeds the {len(correct_indices)} "
            f"available correctly classified images for {network_name}"
        )

    testset = torchvision.datasets.MNIST(
        root=os.path.join(section_dir, "MNIST_Data"),
        train=False,
        download=True,
        transform=transforms.ToTensor(),
    )
    selected = []
    for idx in range(len(testset)):
        if idx in correct_indices:
            image_tensor, _ = testset[idx]
            selected.append((idx, image_tensor.flatten().to(dtype=dtype, device=device)))
            if len(selected) >= num_images:
                break
    return net, selected


def build_forward_precise(pt_mod, net, network_name):
    if network_name == "big":
        def forward_zono_precise(x):
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net[1].weight.T) + net[1].bias)
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net[3].weight.T) + net[3].bias)
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net[5].weight.T) + net[5].bias)
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net[7].weight.T) + net[7].bias)
            x = pt_mod.AffineDualZonotope(x, net[9].weight.T) + net[9].bias
            return x
        return forward_zono_precise

    layers = int(network_name[0])

    def forward_zono_precise(x):
        x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net.fc1.weight.T) + net.fc1.bias.data)
        x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net.fc2.weight.T) + net.fc2.bias.data)
        if layers >= 4:
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net.fc3.weight.T) + net.fc3.bias.data)
        if layers == 5:
            x = pt_mod.PreciseSigmoidDualZonotope(pt_mod.AffineDualZonotope(x, net.fc4.weight.T) + net.fc4.bias.data)
        x = pt_mod.AffineDualZonotope(x, net.fc_final.weight.T) + net.fc_final.bias.data
        return x

    return forward_zono_precise


def eval_splits_once(forward_fn, image, eps_value, n_splits, seed, device, dtype):
    from Duals import DualIntervalTensor
    from SimpleZono import HyperDualIntervalToDualZonotope

    delta = eps_value / n_splits
    one = torch.tensor([1.0], dtype=dtype, device=device)
    sub_bounds = []
    with torch.no_grad():
        for split_idx in range(n_splits):
            l_val = split_idx * delta
            r_val = l_val + delta
            interval_eps = DualIntervalTensor(
                real_l=torch.tensor([l_val], dtype=dtype, device=device),
                real_u=torch.tensor([r_val], dtype=dtype, device=device),
            )
            interval_eps.e1_l[0] = 1
            interval_eps.e1_u[0] = 1
            zono_eps = HyperDualIntervalToDualZonotope(interval_eps)
            hazed_zono = (one - zono_eps) * image + zono_eps
            np.random.seed(seed)
            output = forward_fn(hazed_zono)
            bound_val = torch.max(
                torch.maximum(torch.abs(output.dual.get_lb()), torch.abs(output.dual.get_ub()))
            ).item()
            sub_bounds.append(bound_val)
    return max(sub_bounds)


def main():
    args, device, dtype_name, dtype = parse_args()

    import precise_transformer as pt

    torch.set_default_dtype(dtype)
    pt.USE_VECTORIZED_PRECISE = True
    pt.USE_VECTORIZED_BOUNDARY = True
    pt.USE_BATCHED_LSTSQ = True
    pt.USE_BATCHED_GRID = True
    pt.USE_REAL_CUBIC = args.real_cubic

    eps_schedule = [10 ** (-k / 4) * 2 for k in range(2, 18)]
    all_rows = []

    print(
        f"Split sweep | device={device} | dtype={dtype_name} | real_cubic={args.real_cubic} | "
        f"splits={args.splits} | num_images={args.num_images} | eps_indices={args.eps_indices} | "
        f"warmup={args.warmup} | repeats={args.repeats}"
    )

    for network_name in args.networks:
        net, selected_images = load_network_and_images(
            network_name, device, dtype, args.num_images
        )
        forward_fn = build_forward_precise(pt, net, network_name)

        # Warm-up passes on the first image and first epsilon
        _, first_img = selected_images[0]
        first_eps = eps_schedule[args.eps_indices[0]]
        for _ in range(args.warmup):
            eval_splits_once(forward_fn, first_img, first_eps, 1, args.seed, device, dtype)

        print(f"\n==================== Network: {network_name} ({device}, {dtype_name}) ====================")

        for eps_idx in args.eps_indices:
            eps_val = eps_schedule[eps_idx]
            print(f"\n--- eps_idx={eps_idx} (epsilon={eps_val:.6f}), averaged over {len(selected_images)} images ---")
            print(
                f"{'n_splits':>8s} | {'mean_lc_precise':>16s} | {'reduction_pct':>13s} | "
                f"{'tightening':>11s} | {'time_med_ms':>12s} | {'time_mult':>10s} | "
                f"{'ms_per_split':>12s} | {'peak_vram_mb':>12s}"
            )
            print("-" * 113)

            per_image_split1 = {}
            for image_id, image in selected_images:
                for n_splits in args.splits:
                    times_ms = []
                    peak_mbs = []
                    lc_val = None
                    for _ in range(args.repeats):
                        if device.type == "cuda":
                            torch.cuda.reset_peak_memory_stats(device)
                        t0 = stamp(device)
                        lc_val = eval_splits_once(
                            forward_fn, image, eps_val, n_splits, args.seed, device, dtype
                        )
                        t1 = stamp(device)
                        times_ms.append((t1 - t0) * 1000.0)
                        if device.type == "cuda":
                            peak_mbs.append(torch.cuda.max_memory_allocated(device) / (1024.0 ** 2))
                        else:
                            peak_mbs.append(0.0)

                    t_med = statistics.median(times_ms)
                    t_min = min(times_ms)
                    t_max = max(times_ms)
                    peak_mb = max(peak_mbs)

                    if n_splits == 1:
                        per_image_split1[image_id] = (lc_val, t_med)

                    lc_1, t_1 = per_image_split1[image_id]
                    red_pct = (lc_1 - lc_val) / max(abs(lc_1), 1e-300) * 100.0
                    tight_factor = lc_1 / max(abs(lc_val), 1e-300)
                    t_mult = t_med / max(t_1, 1e-300)

                    all_rows.append(
                        {
                            "network": network_name,
                            "device": str(device),
                            "dtype": dtype_name,
                            "image_id": image_id,
                            "epsilon_index": eps_idx,
                            "epsilon": eps_val,
                            "n_splits": n_splits,
                            "lc_precise": lc_val,
                            "lc_split1": lc_1,
                            "bound_reduction_pct": red_pct,
                            "tightening_factor": tight_factor,
                            "time_median_ms": t_med,
                            "time_min_ms": t_min,
                            "time_max_ms": t_max,
                            "time_split1_median_ms": t_1,
                            "time_multiplier": t_mult,
                            "per_split_ms": t_med / n_splits,
                            "peak_vram_mb": peak_mb,
                        }
                    )

            for n_splits in args.splits:
                sub = [
                    r
                    for r in all_rows
                    if r["network"] == network_name
                    and r["epsilon_index"] == eps_idx
                    and r["n_splits"] == n_splits
                ]
                mean_lc = statistics.mean(r["lc_precise"] for r in sub)
                mean_lc1 = statistics.mean(r["lc_split1"] for r in sub)
                red_pct = (mean_lc1 - mean_lc) / max(abs(mean_lc1), 1e-300) * 100.0
                tight = mean_lc1 / max(abs(mean_lc), 1e-300)
                med_time = statistics.mean(r["time_median_ms"] for r in sub)
                med_time1 = statistics.mean(r["time_split1_median_ms"] for r in sub)
                t_mult = med_time / max(med_time1, 1e-300)
                per_split = med_time / n_splits
                peak_vram = max(r["peak_vram_mb"] for r in sub)
                print(
                    f"{n_splits:8d} | {mean_lc:16.6f} | {red_pct:12.2f}% | "
                    f"{tight:10.2f}x | {med_time:12.2f} | {t_mult:9.2f}x | "
                    f"{per_split:12.2f} | {peak_vram:10.2f}MB"
                )

    if args.out:
        out_dir = os.path.dirname(os.path.abspath(args.out))
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        with open(args.out, "w", newline="") as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=CSV_COLUMNS)
            writer.writeheader()
            writer.writerows(all_rows)
        print(f"\nWrote {len(all_rows)} rows to {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/Pasado/experiments/verify_complex_cubic_fixes.py
"""Demonstrates and tests the two fixes for the complex Cardano cubic solver:

1. Heuristic threshold + residual check:
   - If |Im(x)| <= imag_tol (e.g., 1e-6 or 1e-7), round Im(x) to 0.
   - Plug the real part r = Re(x) back into P(r) = 2*r^3 - 3*r^2 + r - y
     and verify that |P(r)| <= res_tol.

2. Sound floating-point rounding bound (Mine 2004):
   - Propagate floating-point rounding error bounds through Cardano's formula
     and the cubic polynomial evaluation P(r) = 2*r^3 - 3*r^2 + r - y to derive
     per-element thresholds delta_imag(y) and delta_res(r, y) automatically.
"""
import math
import os
import sys
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, os.path.join(ROOT, "forward_mode_tensorized_src"))
sys.path.insert(0, os.path.join(ROOT, "Section_5_4"))

import precise_transformer as pt


def cubic_residual(r, y):
    """Evaluate |2*r^3 - 3*r^2 + r - y| = |sigma''(x) - y| for s = r."""
    return (2.0 * (r ** 3) - 3.0 * (r ** 2) + r - y).abs()


def inverse_poly_tensor_exact_exp(y, complex_dtype=torch.complex64):
    """Same Cardano formula as inverse_poly_tensor, but with 1.0/3.0 instead of 0.33333."""
    Q = torch.tensor(-1.0 / 12.0, dtype=y.dtype, device=y.device)
    R = (0.25 * y).to(complex_dtype)
    J = torch.tensor(1.0j, dtype=complex_dtype, device=y.device)
    sqrt_arg = ((Q ** 3) + (R ** 2)).to(complex_dtype)

    one_third = 1.0 / 3.0
    S = (R + torch.sqrt(sqrt_arg)) ** one_third
    T = (R - torch.sqrt(sqrt_arg)) ** one_third

    x1 = S + T + 0.5
    sqrt_3_by_2 = math.sqrt(3.0) / 2.0
    x2 = -0.5 * (S + T) + 0.5 + (sqrt_3_by_2 * J) * (S - T)
    x3 = -0.5 * (S + T) + 0.5 - (sqrt_3_by_2 * J) * (S - T)
    return (x1, x2, x3)


# ---------------------------------------------------------------------------
# Method 1: Fixed threshold on |Im(x)| + cubic residual check |P(r)| <= res_tol
# ---------------------------------------------------------------------------
def filter_complex_root_fixed_tol(xk, y, imag_tol=1e-6, res_tol=2e-5):
    """Round imaginary part to 0 if |Im(xk)| <= imag_tol, then verify |P(Re(xk))| <= res_tol."""
    r = xk.real.to(y.dtype)
    imag_ok = xk.imag.abs() <= imag_tol
    res_ok = cubic_residual(r, y) <= res_tol
    valid = imag_ok & res_ok
    return torch.where(valid, r, torch.full_like(r, float("nan")))


# ---------------------------------------------------------------------------
# Method 2: Mine (2004) floating-point rounding error interval bounds
# ---------------------------------------------------------------------------
def mine_cardano_rounding_bounds(r, y, complex_dtype=torch.complex64, legacy_exp=True):
    """Compute sound upper bounds (delta_imag, delta_res) from Mine (2004) FP semantics.

    In Mine (2004), each FP operation op(a, b) introduces relative error eps_mach
    plus subnormal floor eta_min:
        |fl(a op b) - (a op b)| <= eps_mach * |a op b| + eta_min

    For Cardano on u^3 - u/4 - y/2 = 0 in the casus irreducibilis (|y| <= 1/(6*sqrt(3))):
      - |S| = |T| = sqrt(|Q|) = 1 / sqrt(12) ~= 0.288675
      - Each complex sqrt, add, pow(1/3), and linear combination accumulates at most
        k_op = 12 elementary FP roundings of size eps_c * |S|.
      - If the legacy exponent 0.33333 is used instead of 1/3, there is an additional
        exponent truncation error: |z^0.33333 - z^(1/3)| <= |1/3 - 0.33333| * |z^(1/3)| * |ln(z)|.
    """
    real_equiv = torch.float32 if complex_dtype == torch.complex64 else torch.float64
    eps_c = torch.finfo(real_equiv).eps          # 1.192e-7 (fp32) or 2.220e-16 (fp64)
    eta_c = torch.finfo(real_equiv).tiny         # subnormal/tiny floor

    # Modulus |S| = |T| = 1 / sqrt(12) for real-root regime
    mod_S = 1.0 / math.sqrt(12.0)

    # 12 elementary operations in Cardano (R, Q^3+R^2, csqrt, R+-W, cpow, S+-T, *sqrt(3)/2)
    k_ops = 12.0
    delta_cardano = k_ops * eps_c * mod_S + eta_c

    if legacy_exp:
        # Legacy code uses x ** 0.33333 instead of 1/3 -> |0.33333 - 1/3| = 1e-5 / 3
        exp_err = abs(1.0 / 3.0 - 0.33333)
        # |ln(S)| = sqrt((0.5*ln(1/12))^2 + theta^2) <= sqrt(1.242^2 + pi^2) ~= 3.38
        delta_exp = exp_err * mod_S * 3.5
    else:
        delta_exp = 0.0

    # 1) Bound on |Im(x_k)| for a true real root:
    delta_imag = 2.0 * delta_cardano

    # 2) Bound on root error |r_tilde - r| = delta_r:
    delta_r = 2.0 * (delta_cardano + delta_exp)

    # 3) Propagate [r - delta_r, r + delta_r] through P(r) = 2*r^3 - 3*r^2 + r - y
    #    using interval arithmetic + per-operation rounding (Mine 2004):
    eps_y = torch.finfo(y.dtype).eps
    abs_r = r.abs()
    poly_cond = 6.0 * (abs_r ** 2) + 6.0 * abs_r + 1.0
    eval_round = 6.0 * eps_y * (2.0 * (abs_r ** 3) + 3.0 * (abs_r ** 2) + abs_r + y.abs())
    delta_res = poly_cond * delta_r + eval_round + eta_c

    return delta_imag, delta_res


def filter_complex_root_mine(xk, y, complex_dtype=torch.complex64, legacy_exp=True):
    """Filter complex root using Mine (2004) FP rounding bounds on Im(xk) and P(Re(xk))."""
    r = xk.real.to(y.dtype)
    delta_imag, delta_res = mine_cardano_rounding_bounds(
        r, y, complex_dtype=complex_dtype, legacy_exp=legacy_exp
    )
    imag_ok = xk.imag.abs() <= delta_imag
    res_ok = cubic_residual(r, y) <= delta_res
    valid = imag_ok & res_ok
    return torch.where(valid, r, torch.full_like(r, float("nan")))


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    l = -1.0 / (6.0 * math.sqrt(3.0))
    u = -l

    # 1000 valid y values inside [l, u] and 200 invalid y values outside [l, u]
    y_valid = torch.linspace(l * 0.9999, u * 0.9999, 1000, dtype=torch.float64, device=device)
    y_invalid = torch.cat([
        torch.linspace(l * 2.0, l * 1.01, 100, dtype=torch.float64, device=device),
        torch.linspace(u * 1.01, u * 2.0, 100, dtype=torch.float64, device=device),
    ])

    if device.type == "cuda":
        # On actual CUDA, complex64 Cardano naturally produces ~1e-8i rounding noise
        x_valid_legacy = list(pt.inverse_poly_tensor(y_valid))
        x_valid_exact = list(inverse_poly_tensor_exact_exp(y_valid))
    else:
        # On CPU, simulate GPU complex64 noise on valid roots (~1e-8 to 4e-8 imaginary residual)
        torch.manual_seed(7)
        gpu_imag_noise = (torch.randn(1000, dtype=torch.float32) * 2e-8).to(torch.complex64) * 1j
        x_valid_legacy = [xk + gpu_imag_noise for xk in pt.inverse_poly_tensor(y_valid)]
        x_valid_exact = [xk + gpu_imag_noise for xk in inverse_poly_tensor_exact_exp(y_valid)]
    x_invalid_legacy = pt.inverse_poly_tensor(y_invalid)

    # Compare how many valid roots (out of 3000) and invalid complex roots are kept:
    orig_kept = sum(torch.isreal(xk).sum().item() for xk in x_valid_legacy)
    m1_legacy_kept = sum(
        torch.isfinite(filter_complex_root_fixed_tol(xk, y_valid, imag_tol=1e-6, res_tol=2e-5)).sum().item()
        for xk in x_valid_legacy
    )
    m1_exact_kept = sum(
        torch.isfinite(filter_complex_root_fixed_tol(xk, y_valid, imag_tol=1e-6, res_tol=1e-6)).sum().item()
        for xk in x_valid_exact
    )
    m2_legacy_kept = sum(
        torch.isfinite(filter_complex_root_mine(xk, y_valid, legacy_exp=True)).sum().item()
        for xk in x_valid_legacy
    )
    m2_exact_kept = sum(
        torch.isfinite(filter_complex_root_mine(xk, y_valid, legacy_exp=False)).sum().item()
        for xk in x_valid_exact
    )

    # On invalid y (outside [-1/(6*sqrt(3)), +1/(6*sqrt(3))]), only 1 root per y is real
    # (outside [0, 1], later rejected by inv_sigmoid) and 2 roots are genuinely complex (400 complex roots).
    m1_false_pos = sum(
        torch.isfinite(filter_complex_root_fixed_tol(xk, y_invalid, imag_tol=1e-6, res_tol=2e-5)).sum().item()
        for xk in x_invalid_legacy[1:]  # x2, x3 are the complex conjugate pair
    )
    m2_false_pos = sum(
        torch.isfinite(filter_complex_root_mine(xk, y_invalid, legacy_exp=True)).sum().item()
        for xk in x_invalid_legacy[1:]
    )

    sample_r = x_valid_legacy[0].real.to(torch.float64)
    d_im_leg, d_res_leg = mine_cardano_rounding_bounds(sample_r, y_valid, legacy_exp=True)
    d_im_ex, d_res_ex = mine_cardano_rounding_bounds(sample_r, y_valid, legacy_exp=False)

    print(f"=== Summary on 3000 valid roots (device={device}) ===")
    print(f"Original torch.isreal(x)            kept: {orig_kept:4d} / 3000")
    print(f"Method 1 (fixed tol, 0.33333)       kept: {m1_legacy_kept:4d} / 3000")
    print(f"Method 1 (fixed tol, 1.0/3.0)       kept: {m1_exact_kept:4d} / 3000")
    print(f"Method 2 (Mine FP bound, 0.33333)   kept: {m2_legacy_kept:4d} / 3000  (delta_imag={d_im_leg:.2e}, max delta_res={d_res_leg.max():.2e})")
    print(f"Method 2 (Mine FP bound, 1.0/3.0)   kept: {m2_exact_kept:4d} / 3000  (delta_imag={d_im_ex:.2e}, max delta_res={d_res_ex.max():.2e})")
    print(f"False positives on 400 true complex roots: Method 1 = {m1_false_pos}, Method 2 = {m2_false_pos}")

    # End-to-end test on 3layer, 4layer, 5layer, big (image 0, eps_idx 0, float64)
    from measure_split_sweep import load_network_and_images, build_forward_precise, eval_splits_once

    torch.set_default_dtype(torch.float64)
    pt.USE_VECTORIZED_PRECISE = True
    pt.USE_VECTORIZED_BOUNDARY = True
    pt.USE_BATCHED_LSTSQ = True
    pt.USE_BATCHED_GRID = True

    eps0 = 10 ** (-2 / 4) * 2
    orig_inv_sig_2nd = pt.inverse_sigmoid_2nd_deriv

    def make_patched_inv_sig(mode):
        def patched(y):
            if mode == "legacy_isreal":
                pt.USE_REAL_CUBIC = False
                return orig_inv_sig_2nd(y)
            if mode == "viete":
                x1, x2, x3 = pt.inverse_poly_real_tensor(y)
                return (pt.inv_sigmoid(x1), pt.inv_sigmoid(x2), pt.inv_sigmoid(x3))
            if mode == "m1_legacy":
                x1, x2, x3 = pt.inverse_poly_tensor(y)
                r1 = filter_complex_root_fixed_tol(x1, y, imag_tol=1e-6, res_tol=2e-5)
                r2 = filter_complex_root_fixed_tol(x2, y, imag_tol=1e-6, res_tol=2e-5)
                r3 = filter_complex_root_fixed_tol(x3, y, imag_tol=1e-6, res_tol=2e-5)
                return (pt.inv_sigmoid(r1), pt.inv_sigmoid(r2), pt.inv_sigmoid(r3))
            if mode == "m1_exact":
                x1, x2, x3 = inverse_poly_tensor_exact_exp(y)
                r1 = filter_complex_root_fixed_tol(x1, y, imag_tol=1e-6, res_tol=1e-6)
                r2 = filter_complex_root_fixed_tol(x2, y, imag_tol=1e-6, res_tol=1e-6)
                r3 = filter_complex_root_fixed_tol(x3, y, imag_tol=1e-6, res_tol=1e-6)
                return (pt.inv_sigmoid(r1), pt.inv_sigmoid(r2), pt.inv_sigmoid(r3))
            if mode == "m2_mine_legacy":
                x1, x2, x3 = pt.inverse_poly_tensor(y)
                r1 = filter_complex_root_mine(x1, y, legacy_exp=True)
                r2 = filter_complex_root_mine(x2, y, legacy_exp=True)
                r3 = filter_complex_root_mine(x3, y, legacy_exp=True)
                return (pt.inv_sigmoid(r1), pt.inv_sigmoid(r2), pt.inv_sigmoid(r3))
            if mode == "m2_mine_exact":
                x1, x2, x3 = inverse_poly_tensor_exact_exp(y)
                r1 = filter_complex_root_mine(x1, y, legacy_exp=False)
                r2 = filter_complex_root_mine(x2, y, legacy_exp=False)
                r3 = filter_complex_root_mine(x3, y, legacy_exp=False)
                return (pt.inv_sigmoid(r1), pt.inv_sigmoid(r2), pt.inv_sigmoid(r3))
            raise ValueError(mode)
        return patched

    print(f"\n=== End-to-End lc_precise across networks (device={device}, image_id=0, eps=0.632456, float64) ===")
    print(
        f"{'Network':8s} | {'Legacy isreal':>14s} | {'Viete (real)':>14s} | "
        f"{'M1 (0.33333)':>14s} | {'M1 (1/3)':>14s} | {'M2 Mine(0.333)':>14s} | {'M2 Mine(1/3)':>14s}"
    )
    print("-" * 103)
    for net_name in ["3layer", "4layer", "5layer", "big"]:
        net, imgs = load_network_and_images(net_name, device, torch.float64, 1)
        _, img = imgs[0]
        fwd = build_forward_precise(pt, net, net_name)
        vals = {}
        for mode in ["legacy_isreal", "viete", "m1_legacy", "m1_exact", "m2_mine_legacy", "m2_mine_exact"]:
            pt.inverse_sigmoid_2nd_deriv = make_patched_inv_sig(mode)
            vals[mode] = eval_splits_once(fwd, img, eps0, 1, 7, device, torch.float64)
        pt.inverse_sigmoid_2nd_deriv = orig_inv_sig_2nd
        print(
            f"{net_name:8s} | {vals['legacy_isreal']:14.6f} | {vals['viete']:14.6f} | {vals['m1_legacy']:14.6f} | "
            f"{vals['m1_exact']:14.6f} | {vals['m2_mine_legacy']:14.6f} | {vals['m2_mine_exact']:14.6f}"
        )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/Pasado/experiments/plot_split_sweep.py
"""Generate clean 2x2 dual-axis plot of bound reduction vs. runtime across --n-splits."""
import argparse
import csv
import statistics
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def load_summary_by_network(csv_path, eps_index=0):
    rows = []
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if int(row["epsilon_index"]) == eps_index:
                rows.append(row)

    networks = ["3layer", "4layer", "5layer", "big"]
    data = {}
    for net in networks:
        net_rows = [r for r in rows if r["network"] == net]
        if not net_rows:
            continue
        splits = sorted({int(r["n_splits"]) for r in net_rows})
        net_summary = []
        for s in splits:
            sub = [r for r in net_rows if int(r["n_splits"]) == s]
            mean_lc = statistics.mean(float(r["lc_precise"]) for r in sub)
            mean_lc1 = statistics.mean(float(r["lc_split1"]) for r in sub)
            mean_t = statistics.mean(float(r["time_median_ms"]) for r in sub)
            net_summary.append(
                {
                    "n_splits": s,
                    "mean_lc": mean_lc,
                    "bound_ratio": mean_lc / mean_lc1,
                    "time_ms": mean_t,
                }
            )
        data[net] = net_summary
    return data


def make_plot(csv_path, out_path):
    data = load_summary_by_network(csv_path, eps_index=0)
    color_bound = "#1f77b4"
    color_time = "#d62728"

    fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=160)
    axes = axes.flatten()

    for idx, net in enumerate(["3layer", "4layer", "5layer", "big"]):
        ax1 = axes[idx]
        ax2 = ax1.twinx()

        d_net = data[net]
        splits = [d["n_splits"] for d in d_net]
        ratios = [d["bound_ratio"] for d in d_net]
        times = [d["time_ms"] for d in d_net]

        line1 = ax1.plot(
            splits, ratios, marker="o", linewidth=2.0, markersize=5,
            color=color_bound, label="Normalized bound"
        )
        line2 = ax2.plot(
            splits, times, marker="s", linewidth=2.0, markersize=5,
            color=color_time, label="Runtime (ms)"
        )

        ax1.set_title(net, fontsize=12, fontweight="bold")
        ax1.set_xlabel("n_splits", fontsize=10)
        ax1.set_ylabel("Normalized bound", color=color_bound, fontsize=10)
        ax2.set_ylabel("Runtime (ms)", color=color_time, fontsize=10)

        ax1.set_ylim(0.0, 1.05)
        ax1.set_xlim(0, max(splits) + 2)
        ax1.set_xticks([1, 8, 16, 32, 64])
        ax1.tick_params(axis="y", labelcolor=color_bound)
        ax2.tick_params(axis="y", labelcolor=color_time)
        ax2.set_ylim(0.0, max(times) * 1.1)
        ax1.grid(True, linestyle=":", alpha=0.5)

        lines = line1 + line2
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc="center right", fontsize=9)

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    make_plot(args.csv, args.out)
    print(f"Saved {args.out}")


## 1. Provera kubnog solvera na CUDA GPU-u (Ideja 1: Prag + Jednačina, Ideja 2: Miné 2004, Viète)


In [ ]:
cmd_cubic = """
set -o pipefail
PYTHONUNBUFFERED=1 python experiments/verify_complex_cubic_fixes.py 2>&1 | tee /content/complex_cubic_fixes_cuda.log
"""
res = subprocess.run(["bash", "-lc", cmd_cubic], cwd=REPO)
assert res.returncode == 0, "Pad pri pokretanju verify_complex_cubic_fixes.py"


## 2. CUDA `--n-splits = 1, 2, 4, 8, 16, 32, 64` (sve 4 mreže + merenje `peak_vram_mb`)


In [ ]:
prefix_cuda = "/content/split_sweep_cuda_fp64_to64"
cmd_cuda = f"""
set -o pipefail
PYTHONUNBUFFERED=1 python experiments/measure_split_sweep.py \
  --networks 3layer 4layer 5layer big \
  --device cuda \
  --dtype float64 \
  --splits 1 2 4 8 16 32 64 \
  --num-images 3 \
  --eps-indices 0 \
  --seed 7 \
  --warmup 2 \
  --repeats 3 \
  --out {prefix_cuda}.csv \
  2>&1 | tee {prefix_cuda}.log
"""
res = subprocess.run(["bash", "-lc", cmd_cuda], cwd=REPO)
assert res.returncode == 0, "Pad pri pokretanju CUDA split sweep-a do 64"


## 3. Tabelarni pregled (uključujući `peak_vram_mb`) i 2x2 Grafik


In [ ]:
import pandas as pd
from IPython.display import Image, display

df_cuda = pd.read_csv("/content/split_sweep_cuda_fp64_to64.csv")
summary = (
    df_cuda[df_cuda["epsilon_index"] == 0]
    .groupby(["network", "n_splits"], as_index=False)
    .agg({
        "lc_precise": "mean",
        "bound_reduction_pct": "mean",
        "tightening_factor": "mean",
        "time_median_ms": "mean",
        "time_multiplier": "mean",
        "per_split_ms": "mean",
        "peak_vram_mb": "max",
    })
    .round(3)
)
display(summary)

subprocess.run([
    "python", "experiments/plot_split_sweep.py",
    "--csv", "/content/split_sweep_cuda_fp64_to64.csv",
    "--out", "/content/split_sweep_dual_axis_cuda_to64.png",
], cwd=REPO, check=True)

display(Image(filename="/content/split_sweep_dual_axis_cuda_to64.png"))


## 4. Preuzimanje svih rezultata i slike (`.zip`)


In [ ]:
import glob
from zipfile import ZipFile
from google.colab import files

paths = [
    "/content/complex_cubic_fixes_cuda.log",
    "/content/split_sweep_cuda_fp64_to64.csv",
    "/content/split_sweep_cuda_fp64_to64.log",
    "/content/split_sweep_dual_axis_cuda_to64.png",
]
archive = "/content/gpu_hybrid_to64_and_cubic_fixes.zip"

with ZipFile(archive, "w") as output:
    for path in paths:
        if os.path.exists(path):
            output.write(path, arcname=os.path.basename(path))

print("Arhivirani fajlovi:", [os.path.basename(p) for p in paths if os.path.exists(p)])
files.download(archive)
